In [1]:
from bs4 import BeautifulSoup
import requests
import lxml
import re

import pandas as pd

from fake_useragent import UserAgent
ua = UserAgent()

In [2]:
month = '03'

In [3]:
year = '2022'

In [4]:
def get_soup(url, timeout=5):
    headers  = {'User-Agent':ua.random}
    try:
        response = requests.get(url,headers=headers)
    except:
        print("FAILED "+ url)
        return 0
    attempts = 0
    while(not response.ok):
            print((url+' failed with code: '+str(response.status_code)))
            if attempts > timeout:
                print(url+' failed with code: '+str(response.status_code))
                return BeautifulSoup('')
            response = requests.get(url)
            attempts += 1
    page = response.text
    soup = BeautifulSoup(page)
    return soup

In [5]:
url = 'http://sumodb.sumogames.de/Banzuke.aspx?b='+year+month+'&heya=-1&shusshin=-1&ho=on&c=on&simple=on'

In [6]:
soup = get_soup(url)

In [7]:
rows = []
for child in soup.find_all('table',{"class":"banzuke"})[0].children:
    count = 1
    for td in child:
        if count<2:
            count += 1
            continue
        row = []
        for tr in td:
            row.append(tr)
        rows.append(row)

In [8]:
records = pd.DataFrame.from_records(rows)

In [9]:
records.columns = ['X','Rank','Rikishi','Hoshitori','W','L','Y']

In [10]:
records = records.iloc[2:-1,:]

In [11]:
def row_clean(row):
    row['Rank'] = row['Rank'].text
    row['Rikishi'] = row['Rikishi'].text
    row['W'] = row['W'].text
    row['L'] = row['L'].text
    return row

In [12]:
standings = records.apply(row_clean,axis=1)
standings.Hoshitori = records.Hoshitori.map(lambda x: x.text)

In [13]:
standings = standings[['Rank','Rikishi','Hoshitori','W','L']]

In [14]:
standings.to_csv('current_makuuchi_standings.csv',index=False)